# **Data Loading & Exploration**



1.   Loading the datset
2.   Getting the shape of the dataset
3.   Analyzing class distribution


In [5]:
import re
import string
import pandas as pd
import numpy as np

# 1.
df = pd.read_csv('/content/sample_data/tweets.csv')
# 2.
print(f"Dataset shape: {df.shape}")
print(df.head())
# 3.
print("\nClass distribution:")
print(df['label'].value_counts(normalize=True))

Dataset shape: (7920, 3)
   id  label                                              tweet
0   1      0  #fingerprint #Pregnancy Test https://goo.gl/h1...
1   2      0  Finally a transparant silicon case ^^ Thanks t...
2   3      0  We love this! Would you go? #talk #makememorie...
3   4      0  I'm wired I know I'm George I was made that wa...
4   5      1  What amazing service! Apple won't even talk to...

Class distribution:
label
0    0.744192
1    0.255808
Name: proportion, dtype: float64


# **Text Preprocessing & Cleaning**



1.   Converting to lowercase
2.   Removing URLs
3.   Removing HTML tags
4.   Removing @ and # symbols while retaining the text
5.   Removing punctuation
6.   Normalizing whitespaces
7.   Applying cleaning function to the dataset

In [6]:
def clean_tweet(text):
    # 1.
    text = text.lower()
    # 2.
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    # 3.
    text = re.sub(r'<.*?>', '', text)
    # 4.
    text = re.sub(r'[@#]', '', text)
    # 5.
    text = re.sub(r'[%s]' % re.escape(string.punctuation), '', text)
    # 6.
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# 7.
df['clean_tweet'] = df['tweet'].apply(clean_tweet)

# **Dataset Splitting**



1.   80-20 stratified train-test split




In [7]:
from sklearn.model_selection import train_test_split

X = df['clean_tweet']
y = df['label']

# 1.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f"Training samples: {len(X_train)} | Test samples: {len(X_test)}")

Training samples: 6336 | Test samples: 1584


# **Feature Extraction**



1.   TF-IDF Vectorization
2.   Vectorizing text using unigrams and bigrams
3.   Fitting on training data and transforming both sets


In [8]:
# 1.
from sklearn.feature_extraction.text import TfidfVectorizer

# 2.
vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    stop_words='english'
)

# 3.
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

# **Model Training & Evaluation**



1.   Initializing and training the classifier
2.   Predicting on unseen test data
3.   Evaluation metrics


In [9]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# 1.
model = LogisticRegression(C=2.0, max_iter=1000, random_state=42)
model.fit(X_train_tfidf, y_train)

# 2.
y_pred = model.predict(X_test_tfidf)

# 3.
acc = accuracy_score(y_test, y_pred)
print(f"Test Accuracy: {acc * 100:.2f}%\n")
print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=['Positive (0)', 'Negative (1)']))

Test Accuracy: 89.33%

Classification Report:
              precision    recall  f1-score   support

Positive (0)       0.91      0.96      0.93      1179
Negative (1)       0.85      0.71      0.77       405

    accuracy                           0.89      1584
   macro avg       0.88      0.83      0.85      1584
weighted avg       0.89      0.89      0.89      1584



# **Predicting Sentiment on Custom Tweets**



1.   Normalizing the input
2.   Text preprocessing
3.   Vectorization
4.   Making Predictions & Calculating Confidence
5.   Mapping and Formatting the Output


In [13]:
def predict_sentiment(tweets):
    # 1.
    if isinstance(tweets, str):
        tweets = [tweets]
    elif hasattr(tweets, 'tolist'):
        tweets = tweets.tolist()

    # 2.
    cleaned = [clean_tweet(t) for t in tweets]

    # 3.
    features = vectorizer.transform(cleaned)

    # 4.
    predictions = model.predict(features)
    probabilities = model.predict_proba(features)

    # 5.
    labels = {0: 'Positive', 1: 'Negative'}

    for tweet, pred, prob in zip(tweets, predictions, probabilities):
        confidence = prob[pred] * 100
        print(f"[{labels[pred]}] ({confidence:.1f}% confidence) -> {tweet}")

In [15]:
predict_sentiment(["Great battery life", "Terrible screen quality"])

[Positive] (59.6% confidence) -> Great battery life
[Negative] (59.8% confidence) -> Terrible screen quality


In [16]:
predict_sentiment("Phone is getting heated so bad #NotHappy")

[Negative] (57.7% confidence) -> Phone is getting heated so bad #NotHappy
